## **Setting up Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Defining constants (location) of the storage locations
PROJECT_ROOT = "/content/drive/MyDrive/aviation-delay-analytics-notebooks"
BRONZE_PARQUET_PATH = f"{PROJECT_ROOT}/data/01_bronze/flights_2009_parquet"
SILVER_PARQUET_PATH = f"{PROJECT_ROOT}/data/02_silver/flights_2009_clean"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Ingestion into the Instance

In [ ]:
import os
from google.colab import userdata

# Enter your legacy token details here
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Download the data
DATASET = "yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018"
!kaggle datasets download -d {DATASET}


Dataset URL: https://www.kaggle.com/datasets/yuanyuwendymu/airline-delay-and-cancellation-data-2009-2018
License(s): other
airline-delay-and-cancellation-data-2009-2018.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# As the data from API is in zip, we unzip into the memory
!unzip airline-delay-and-cancellation-data-2009-2018.zip

Archive:  airline-delay-and-cancellation-data-2009-2018.zip
replace 2009.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2010.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2011.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2012.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2013.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2014.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2015.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2016.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2017.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace 2018.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


Data Exploration

In [ ]:
# Taking one csv as sample and checking data
import pandas as pd

sample = pd.read_csv("2009.csv", nrows=5)

sample.head()

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,Unnamed: 27
0,2009-01-01,XE,1204,DCA,EWR,1100,1058.0,-2.0,18.0,1116.0,...,62.0,68.0,42.0,199.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-01-01,XE,1206,EWR,IAD,1510,1509.0,-1.0,28.0,1537.0,...,82.0,75.0,43.0,213.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-01-01,XE,1207,EWR,DCA,1100,1059.0,-1.0,20.0,1119.0,...,70.0,62.0,36.0,199.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-01-01,XE,1208,DCA,EWR,1240,1249.0,9.0,10.0,1259.0,...,77.0,56.0,37.0,199.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-01-01,XE,1209,IAD,EWR,1715,1705.0,-10.0,24.0,1729.0,...,105.0,77.0,40.0,213.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   FL_DATE              5 non-null      object 
 1   OP_CARRIER           5 non-null      object 
 2   OP_CARRIER_FL_NUM    5 non-null      int64  
 3   ORIGIN               5 non-null      object 
 4   DEST                 5 non-null      object 
 5   CRS_DEP_TIME         5 non-null      int64  
 6   DEP_TIME             5 non-null      float64
 7   DEP_DELAY            5 non-null      float64
 8   TAXI_OUT             5 non-null      float64
 9   WHEELS_OFF           5 non-null      float64
 10  WHEELS_ON            5 non-null      float64
 11  TAXI_IN              5 non-null      float64
 12  CRS_ARR_TIME         5 non-null      int64  
 13  ARR_TIME             5 non-null      float64
 14  ARR_DELAY            5 non-null      float64
 15  CANCELLED            5 non-null      float64

In [ ]:
# Verifying number of features to ensure combining dataset won't have issues
import pandas as pd

for year in range(2009, 2019):
    df = pd.read_csv(f"{year}.csv", nrows=5)
    print(year, len(df.columns))

2009 28
2010 28
2011 28
2012 28
2013 28
2014 28
2015 28
2016 28
2017 28
2018 28


In [ ]:
reference = pd.read_csv("2009.csv", nrows=0).columns

for year in range(2010, 2019):
    cols = pd.read_csv(f"{year}.csv", nrows=0).columns

    print(year, reference.equals(cols))

2010 True
2011 True
2012 True
2013 True
2014 True
2015 True
2016 True
2017 True
2018 True


### **Starting Spark Operations**

In [ ]:
# Creating SparkSession object
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Aviation Analytics") \
    .getOrCreate()

In [ ]:
# Creating the Schema, as inferSchema takes too long to process
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

flight_schema = StructType([
    StructField("FL_DATE", StringType(), True), # Read as String first to prevent parse errors, convert in Silver
    StructField("OP_CARRIER", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("DEST", StringType(), True),
    StructField("CRS_DEP_TIME", DoubleType(), True),
    StructField("DEP_TIME", DoubleType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("WHEELS_OFF", DoubleType(), True),
    StructField("WHEELS_ON", DoubleType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", DoubleType(), True),
    StructField("ARR_TIME", DoubleType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("CANCELLATION_CODE", StringType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("CARRIER_DELAY", DoubleType(), True),
    StructField("WEATHER_DELAY", DoubleType(), True),
    StructField("NAS_DELAY", DoubleType(), True),
    StructField("SECURITY_DELAY", DoubleType(), True),
    StructField("LATE_AIRCRAFT_DELAY", DoubleType(), True),
    StructField("Unnamed: 27", StringType(), True) # Capture the phantom column
])

Taking 2009 Dataset as Sample

In [ ]:
from pyspark.sql import SparkSession

# Creating DataFrame using Spark
df_raw = spark.read.csv(
    "2009.csv",
    header=True,
    schema=flight_schema
)

In [ ]:
# Removing trailing column
df_cleaned = df_raw \
    .drop("Unnamed: 27")
df_cleaned

DataFrame[FL_DATE: string, OP_CARRIER: string, OP_CARRIER_FL_NUM: int, ORIGIN: string, DEST: string, CRS_DEP_TIME: double, DEP_TIME: double, DEP_DELAY: double, TAXI_OUT: double, WHEELS_OFF: double, WHEELS_ON: double, TAXI_IN: double, CRS_ARR_TIME: double, ARR_TIME: double, ARR_DELAY: double, CANCELLED: double, CANCELLATION_CODE: string, DIVERTED: double, CRS_ELAPSED_TIME: double, ACTUAL_ELAPSED_TIME: double, AIR_TIME: double, DISTANCE: double, CARRIER_DELAY: double, WEATHER_DELAY: double, NAS_DELAY: double, SECURITY_DELAY: double, LATE_AIRCRAFT_DELAY: double]

In [ ]:
df_cleaned.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: doub

In [ ]:
df_cleaned.count()

6429338

## **Passing on the Dataframe**

Creating Folder Structure for Bronze Data Storage

In [ ]:
%%bash

PROJECT_ROOT="/content/drive/MyDrive/aviation-delay-analytics-notebooks"

# Cehck if already exist, else create folder
if [ ! -d "${PROJECT_ROOT}/data/01_bronze" ]; then
  mkdir -p "${PROJECT_ROOT}/data/01_bronze"
  echo "Created Bronze storage folder on Drive."
else
  echo "Bronze storage folder already exists on Drive."
fi

Bronze storage folder already exists on Drive.


Write the DataFrame into Paraquet in the Location

In [ ]:
# Save the structured dataframe to the Bronze Parquet zone on Drive
print(f"Committing immutable staging logs to Bronze Parquet format at: {BRONZE_PARQUET_PATH}")

df_cleaned.write \
    .mode("overwrite") \
    .parquet(BRONZE_PARQUET_PATH)

print("🎉 Success! Raw Ingestion Complete.")

💾 Committing immutable staging logs to Bronze Parquet format at: /content/drive/MyDrive/aviation-delay-analytics-notebooks/data/01_bronze/flights_2009_parquet
🎉 Success! Raw Ingestion Complete.


Verify Bronze Parquet

In [ ]:
%%bash
# Verify the generated parquet folder on Google Drive
PARQUET_ZONE="/content/drive/MyDrive/aviation-delay-analytics-notebooks/data/01_bronze/flights_2009_parquet"

echo "🔎 Reviewing generation artifacts in Bronze Parquet Target..."

if [ -d "$PARQUET_ZONE" ]; then
    echo "🟢 Parquet directory integrity verified on Google Drive! Listing top files:"
    ls -lh "$PARQUET_ZONE" | head -n 6
else
    echo "❌ Execution Failure: Target Parquet zone is empty or corrupted."
fi

🔎 Reviewing generation artifacts in Bronze Parquet Target...
🟢 Parquet directory integrity verified on Google Drive! Listing top files:
total 139M
-rw------- 1 root root 24M Jul 15 09:45 part-00000-f1882ea7-9466-450d-a927-665ff14c7264-c000.snappy.parquet
-rw------- 1 root root 24M Jul 15 09:45 part-00001-f1882ea7-9466-450d-a927-665ff14c7264-c000.snappy.parquet
-rw------- 1 root root 24M Jul 15 09:45 part-00002-f1882ea7-9466-450d-a927-665ff14c7264-c000.snappy.parquet
-rw------- 1 root root 24M Jul 15 09:45 part-00003-f1882ea7-9466-450d-a927-665ff14c7264-c000.snappy.parquet
-rw------- 1 root root 24M Jul 15 09:46 part-00004-f1882ea7-9466-450d-a927-665ff14c7264-c000.snappy.parquet
